# UC-02 Tutorial: Report-Ready Dataset

This tutorial runs `run_use_case_02_report_ready_dataset.py`.

Goal:
- Start from UC-01 baseline
- Add report-oriented entities (ML models, evaluations, timeline states, extra emotions)
- Populate the UI report dashboards with richer data

Prerequisites:
- Open this notebook from inside the SEGB repository.
- Python env:
  - `./.venv/bin/python -m pip install -e packages/semantic_log_generator`
- To publish data or validate UI views, start backend API at `http://localhost:5000`.
- Run the backend preflight cell below before executing the main use-case cell.
- UI routes can be opened on:
  - Production: `http://localhost:8080`
  - Development: `http://localhost:5173`


In [ ]:
# Ensure repository root is available in sys.path
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'examples').exists() and (candidate / 'packages' / 'semantic_log_generator').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError('Repository root not found. Open this notebook inside the SEGB repository.')

extra_paths = [
    repo_root,
    repo_root / 'packages' / 'semantic_log_generator' / 'src',
    repo_root / 'apps' / 'backend' / 'src',
]
for path in extra_paths:
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f'Using repo root: {repo_root}')


In [ ]:
# Backend preflight (health checks)
import json
import urllib.error
import urllib.request

API_URL = 'http://localhost:5000'
REQUIRE_BACKEND = False


def _get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=5) as response:
        payload = response.read().decode('utf-8')
    return json.loads(payload)


def check_backend_health(api_url: str, *, require: bool) -> None:
    base = api_url.rstrip('/')
    live_url = f"{base}/healthz/live"
    ready_url = f"{base}/healthz/ready"

    try:
        live = _get_json(live_url)
        ready = _get_json(ready_url)
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        message = f"Backend not reachable at {api_url}: {exc}"
        if require:
            raise RuntimeError(message) from exc
        print('WARNING:', message)
        return

    live_ok = bool(live.get('live'))
    ready_ok = bool(ready.get('ready')) and bool(ready.get('virtuoso'))

    print('healthz/live:', live)
    print('healthz/ready:', ready)

    if not (live_ok and ready_ok):
        message = f"Backend unhealthy at {api_url} (live_ok={live_ok}, ready_ok={ready_ok})"
        if require:
            raise RuntimeError(message)
        print('WARNING:', message)
        return

    print(f"Backend preflight OK -> {api_url}")


check_backend_health(API_URL, require=REQUIRE_BACKEND)


In [ ]:
from examples.simulations.run_use_case_02_report_ready_dataset import run_report_ready_simulation

result = run_report_ready_simulation()
graph = result.graph

print('Use case: UC-02 report-ready dataset')
print('Base shared event:', result.base_result.shared_event_uri)
print('Base timestamp:', result.base_timestamp.isoformat())
print('Triples generated:', len(graph))


In [ ]:
# Quick sanity check: how many emotion annotations exist?
from rdflib.namespace import RDF
from semantic_log_generator.namespaces import AMOR

emotion_annotations = list(graph.subjects(RDF.type, AMOR.EmotionAnnotation))
print('Emotion annotations:', len(emotion_annotations))


## Optional: Publish to Backend

```bash
./.venv/bin/python -m examples.simulations.run_use_case_02_report_ready_dataset \
  --publish-url http://localhost:5000 \
  --no-print-ttl
```

## What to Verify in the Web UI

Use the same route on prod (`http://localhost:8080`) or dev (`http://localhost:5173`).

1. `/reports`:
- Participants report should show Maria and both robots.
- Emotion timeline should show fear and later happiness.
- Movement/displacement report should show ARI and TIAGo states.

2. `/kg-graph`:
- Inspect links among listening/decision/response/emotion activities.

3. `/system/logs`:
- Confirm backend recorded TTL insert operations.
